# Taller: Arquitectura de Datos con Window Functions y Joins Optimizados

**Contexto:** Como Científicos de Datos, procesar volúmenes masivos de datos (NYC Taxi Dataset) requiere que dominemos no solo la estadística, sino la eficiencia del clúster.

--- 
### 📖 Referencia Técnica Obligatoria
Para resolver este taller, **debes leer y consultar** constantemente la siguiente guía técnica sobre Window Functions:
👉 [Window Functions in PySpark (Medium)](https://medium.com/@roshmitadey/window-functions-in-pyspark-2f342e8b9805)

--- 
### Objetivos del Taller:
1. Diferenciar Agregaciones Globales (`groupBy`) de Agregaciones Segmentadas (`Window`).
2. Implementar funciones de Ranking, Agregación y Analíticas.
3. Comprender el costo del *Shuffle* y la potencia del *Broadcast Join*.
4. Aplicar buenas prácticas de almacenamiento columnar (Parquet).

In [1]:
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F
import os

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Taller_NYC_Taxi_Expert') \
    .getOrCreate()

print('Sesión de Spark iniciada. El clúster está listo para la acción.')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/12 01:13:55 WARN Utils: Your hostname, codespaces-1e64b7, resolves to a loopback address: 127.0.0.1; using 10.0.0.24 instead (on interface eth0)
26/05/12 01:13:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 01:13:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Sesión de Spark iniciada. El clúster está listo para la acción.


## EJERCICIO 1: Ingesta Profesional (Parquet vs CSV)
**Teoría:** En Big Data, los archivos CSV son ineficientes porque son planos. **Parquet** es un formato columnar que permite a Spark leer solo las columnas necesarias, ahorrando hasta un 80% de tiempo de I/O.

**Tarea:** Descarga el dataset oficial (250MB) y cárgalo directamente.

In [ ]:
if not os.path.exists('taxis.parquet'):
    os.system('wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2015-01.parquet -O taxis.parquet')

### INICIA TU CÓDIGO AQUÍ ###
df_raw = spark.read.____('taxis.parquet')
### TERMINA TU CÓDIGO AQUÍ ###

df_taxis = df_raw.select('VendorID', 'tpep_pickup_datetime', 'PULocationID', 'tip_amount', 'total_amount')
print(f'Registros cargados: {df_taxis.count()}')

## EJERCICIO 2: El Límite de GroupBy (Agregación con pérdida)
**Teoría:** `groupBy` es una operación de resumen. Reduce el número de filas al agruparlas por una clave. Es útil para reportes finales, pero destructivo si necesitas mantener el detalle del dato original.

**Tarea:** Calcula el promedio de propina (`tip_amount`) por zona (`PULocationID`).

In [ ]:
### INICIA TU CÓDIGO AQUÍ ###
df_resumen = df_taxis.____('PULocationID').____('tip_amount')
### TERMINA TU CÓDIGO AQUÍ ###

print('Observa cómo la tabla se redujo a una fila por zona:')
df_resumen.show(5)

## EJERCICIO 3: Definición de la Ventana
**Teoría:** Para usar Window Functions, primero debemos definir la "especificación de la ventana". Esto le dice a Spark cómo particionar los datos antes de aplicar el cálculo.

**Tarea:** Crea una especificación de ventana que particione los datos por zona de recogida (`PULocationID`).

In [ ]:
### INICIA TU CÓDIGO AQUÍ ###
ventana_zona = Window.____('PULocationID')
### TERMINA TU CÓDIGO AQUÍ ###
print('Ventana definida. Aún no se ha ejecutado ningún cálculo (Lazy Evaluation).')

## EJERCICIO 4: Aggregate Window Function (avg)
**Teoría:** A diferencia del `groupBy`, aquí el cálculo se realiza pero se "pega" a cada fila original.

**Tarea:** Crea una columna llamada `media_zona` que contenga el promedio de propina de su zona, usando la ventana definida anteriormente.

In [ ]:
### INICIA TU CÓDIGO AQUÍ ###
df_final = df_taxis.withColumn('media_zona', F.____('tip_amount').____(ventana_zona))
### TERMINA TU CÓDIGO AQUÍ ###

df_final.select('PULocationID', 'tip_amount', 'media_zona').show(5)

## EJERCICIO 5: Ranking Functions (row_number)
**Teoría:** Las funciones de ranking requieren que la ventana esté **ordenada**. `row_number()` asigna un ID secuencial del 1 al N dentro de cada partición.

**Tarea:** Enumera los viajes de cada zona (`PULocationID`) del más caro al más barato (`total_amount`).

In [ ]:
### INICIA TU CÓDIGO AQUÍ ###
ventana_ranking = Window.partitionBy('PULocationID').____(F.col('total_amount').____())

df_rankeado = df_taxis.withColumn('num_viaje', F.____().over(ventana_ranking))
### TERMINA TU CÓDIGO AQUÍ ###

df_rankeado.select('PULocationID', 'total_amount', 'num_viaje').show(10)

## EJERCICIO 6: Rank vs Dense_Rank
**Teoría:** ¿Qué pasa con los empates? `rank` deja huecos (ej: 1, 2, 2, 4) y `dense_rank` no (ej: 1, 2, 2, 3).

**Tarea:** Aplica ambas funciones y observa la diferencia en viajes con el mismo costo.

In [ ]:
### INICIA TU CÓDIGO AQUÍ ###
df_empates = df_taxis.withColumn('rank', F.____().over(ventana_ranking)) \
                     .withColumn('dense_rank', F.____().over(ventana_ranking))
### TERMINA TU CÓDIGO AQUÍ ###

df_empates.select('PULocationID', 'total_amount', 'rank', 'dense_rank').show(10)

## EJERCICIO 7: Analytic Functions (lag)
**Teoría:** `lag` permite ver el pasado. Accede a una fila previa dentro de la misma partición.

**Tarea:** Para cada taxi (`VendorID`), calcula la diferencia de monto entre el viaje actual y el anterior.

In [ ]:
### INICIA TU CÓDIGO AQUÍ ###
ventana_tiempo = Window.partitionBy('VendorID').orderBy('tpep_pickup_datetime')

df_analitico = df_taxis.withColumn('monto_previo', F.____('total_amount').over(ventana_tiempo)) \
                       .withColumn('diferencia', F.col('total_amount') - F.col('monto_previo'))
### TERMINA TU CÓDIGO AQUÍ ###

df_analitico.select('VendorID', 'total_amount', 'monto_previo', 'diferencia').show(10)

## EJERCICIO 8: El Costo del Shuffle (Join Estándar)
**Teoría:** Cruzar dos tablas masivas obliga a Spark a mover datos por la red para encontrar las coincidencias. Esto se llama **Shuffle** y es el enemigo número 1 del rendimiento.

**Tarea:** Realiza un Join tradicional entre los millones de taxis y una pequeña lista de zonas VIP.

In [ ]:
# Tabla pequeña de zonas VIP
zonas_vip = [(1, 'JFK Airport'), (2, 'Wall Street'), (138, 'La Guardia Airport')]
df_vip = spark.createDataFrame(zonas_vip, ['id', 'nombre_lugar'])

### INICIA TU CÓDIGO AQUÍ ###
# Realiza un Join estándar (Inner)
df_estandar = df_taxis.____(df_vip, df_taxis.PULocationID == df_vip.____, 'inner')
### TERMINA TU CÓDIGO AQUÍ ###

df_estandar.show(5)

## EJERCICIO 9: Optimización de Red (Broadcast Join)
**Teoría:** Si sabemos que una tabla es minúscula (KB o MB) comparada con la principal (GB), podemos enviarla a la memoria de todos los nodos. Así, cada nodo hace el Join localmente y **eliminamos el Shuffle**.

**Tarea:** Implementa el mismo Join anterior pero usando la función `broadcast`.

In [ ]:
### INICIA TU CÓDIGO AQUÍ ###
df_broadcast = df_taxis.join(F.____(df_vip), df_taxis.PULocationID == df_vip.id)
### TERMINA TU CÓDIGO AQUÍ ###

df_broadcast.select('tpep_pickup_datetime', 'nombre_lugar').show(5)

## EJERCICIO 10: Reto Final Combinado
**Tarea:** Queremos el nombre del lugar VIP (Broadcast Join) pero solo para los viajes que son el número 1 en costo (row_number) dentro de su zona.

**Restricción:** Debes usar una ventana y un join optimizado.

In [ ]:
### INICIA TU CÓDIGO AQUÍ ###
# 1. Aplica el ranking
df_ranking_final = df_taxis.withColumn('pos', F.row_number().over(ventana_ranking))

# 2. Filtra el top 1
df_top1 = df_ranking_final.____(F.col('pos') == 1)

# 3. Haz el join con el catálogo VIP (Optimizado)
df_resultado = df_top1.join(F.____(df_vip), df_top1.PULocationID == df_vip.id)
### TERMINA TU CÓDIGO AQUÍ ###

df_resultado.select('nombre_lugar', 'total_amount').show()